<a href="https://colab.research.google.com/github/KhushiAayat/E-Commerce-Analytics/blob/main/E_Commerce_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import os


In [3]:
output_dir = 'clean_data'
os.makedirs(output_dir, exist_ok=True)
#directory to store all clean data

EXTRACT

In [4]:
df_customers=pd.read_csv('/content/olist_customers_dataset.csv')
df_items=pd.read_csv('/content/olist_order_items_dataset.csv')
df_payments=pd.read_csv('/content/olist_order_payments_dataset.csv')
df_orders=pd.read_csv('/content/olist_orders_dataset.csv')
df_products=pd.read_csv('/content/olist_products_dataset.csv')
df_translation=pd.read_csv('/content/product_category_name_translation.csv')

Transform(clean individual table)

In [5]:
clean_customers=df_customers.drop_duplicates()
print('clean_customers , No duplicates')

clean_customers , No duplicates


In [6]:
clean_products=  pd.merge(df_products, df_translation, on='product_category_name', how='left')
clean_products.drop(columns=['product_category_name'],inplace=True)
clean_products.rename(columns={'product_category_name_english':'product_category'},inplace=True)
clean_products['product_category']=clean_products['product_category'].fillna('unknown')
for col in['product_name_lenght','product_description_lenght','product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
  clean_products[col]=clean_products[col].fillna(clean_products[col].median())

print("Clean Products")

Clean Products


In [7]:
clean_payments=df_payments.drop_duplicates()
clean_payments['payment_value']=clean_payments['payment_value'].astype(float)
clean_payments=clean_payments[clean_payments['payment_value']>0]

In [12]:
clean_orders = df_orders.drop_duplicates()
clean_orders = clean_orders[clean_orders['order_status'] == 'delivered']

datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in datetime_cols:
    clean_orders[col] = pd.to_datetime(clean_orders[col], errors='coerce')
clean_orders.dropna(subset=['order_purchase_timestamp', 'order_delivered_customer_date'], inplace=True)



Transform (building master table)

In [18]:
master_df=clean_orders.copy()
master_df=pd.merge(master_df, df_items, on='order_id', how='left')
master_df=pd.merge(master_df, clean_products[['product_id', 'product_category']], on='product_id', how='left')
master_df = pd.merge(master_df, clean_customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']], on='customer_id', how='left')

Feature


In [20]:
master_df['total_item_value']=master_df['price']+master_df['freight_value']
master_df['delivery_delay_days']=(master_df['order_delivered_customer_date']-master_df['order_estimated_delivery_date']).dt.days
master_df['purshase_year']=master_df['order_purchase_timestamp'].dt.year
master_df['purchase_month']=master_df['order_purchase_timestamp'].dt.month
master_df['purchase_day_of_week']=master_df['order_purchase_timestamp'].dt.day_name()


LOAD: export the cleaned data


In [21]:
clean_customers.to_csv(f'{output_dir}/clean_customers.csv',index=False)
clean_products.to_csv(f'{output_dir}/clean_products.csv', index=False)
clean_payments.to_csv(f'{output_dir}/clean_payments.csv', index=False)
clean_orders.to_csv(f'{output_dir}/clean_orders.csv',index=False)
master_df.to_csv(f'{output_dir}/analytics_master_table.csv', index=False)
print('pipeline complete')

pipeline complete
